In [1]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
import reverse_geocoder as rg
from datetime import date, timedelta

StatementMeta(, 7a11701f-4e7c-44d8-a457-8c62031a95c0, 5, Finished, Available, Finished, False)

In [2]:
# Remove this line before running Data Factory Pipeline
start_date = date.today() - timedelta(7)

StatementMeta(, 7a11701f-4e7c-44d8-a457-8c62031a95c0, 6, Finished, Available, Finished, False)

In [3]:
df = spark.read.table("earthquake_events_silver").filter(col("time") > start_date)

StatementMeta(, 7a11701f-4e7c-44d8-a457-8c62031a95c0, 7, Finished, Available, Finished, False)

In [4]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude & longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using reverse geocoder API.
    """
    coordinates = (float(lat), float(lon))
    return rg.search(coordinates)[0].get("cc")

StatementMeta(, 7a11701f-4e7c-44d8-a457-8c62031a95c0, 8, Finished, Available, Finished, False)

In [5]:
# Registering udf so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType()) 

StatementMeta(, 7a11701f-4e7c-44d8-a457-8c62031a95c0, 9, Finished, Available, Finished, False)

In [6]:
# Adding country code and city attributes
df_with_location = \
                df.\
                    withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

StatementMeta(, 7a11701f-4e7c-44d8-a457-8c62031a95c0, 10, Finished, Available, Finished, False)

In [7]:
# Adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn("sig_class",
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )                                         

StatementMeta(, 7a11701f-4e7c-44d8-a457-8c62031a95c0, 11, Finished, Available, Finished, False)

In [9]:
# Appending to the gold table
df_with_location_sig_class.write.mode("append").saveAsTable("earthquake_events_gold")

StatementMeta(, 7a11701f-4e7c-44d8-a457-8c62031a95c0, 13, Finished, Available, Finished, False)